<a href="https://colab.research.google.com/github/OlajideFemi/Carbon-Footprint/blob/main/Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
Simple sentiment analysis model.

Usage:
    from sentiment import analyze_sentiment
    import pandas as pd

    df = pd.DataFrame({"text": ["I love this!", "This is terrible.", "It's okay."]})
    result = analyze_sentiment(df, text_col="text")
    print(result)

Approach:
  - Lexicon-based scoring via VADER (works well on short/free text, no training needed).
  - Falls back to a tiny built-in lexicon if `vaderSentiment` isn't installed.
  - Returns the original df with added columns: neg, neu, pos, compound, sentiment.
"""

from __future__ import annotations
import re
import pandas as pd


def _get_analyzer():
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        return SentimentIntensityAnalyzer()
    except ImportError:
        return _FallbackAnalyzer()


class _FallbackAnalyzer:
    """Tiny lexicon fallback if vaderSentiment isn't installed."""
    POS = {"good","great","love","excellent","amazing","awesome","happy","best",
           "fantastic","wonderful","nice","like","enjoy","perfect","superb"}
    NEG = {"bad","terrible","hate","awful","worst","horrible","sad","poor",
           "disappointing","boring","ugly","dislike","broken","angry","annoying"}

    def polarity_scores(self, text: str):
        tokens = re.findall(r"[a-zA-Z']+", str(text).lower())
        if not tokens:
            return {"neg":0.0,"neu":1.0,"pos":0.0,"compound":0.0}
        pos = sum(t in self.POS for t in tokens)
        neg = sum(t in self.NEG for t in tokens)
        total = len(tokens)
        p, n = pos/total, neg/total
        neu = max(0.0, 1.0 - p - n)
        compound = max(-1.0, min(1.0, (pos - neg) / (pos + neg + 1)))
        return {"neg":n,"neu":neu,"pos":p,"compound":compound}


def _label(compound: float) -> str:
    if compound >= 0.05:
        return "positive"
    if compound <= -0.05:
        return "negative"
    return "neutral"


def analyze_sentiment(df: pd.DataFrame, text_col: str = "text") -> pd.DataFrame:
    """Score each row's free text; returns df + neg/neu/pos/compound/sentiment."""
    if text_col not in df.columns:
        raise KeyError(f"Column '{text_col}' not found. Available: {list(df.columns)}")

    analyzer = _get_analyzer()
    scores = df[text_col].fillna("").astype(str).apply(analyzer.polarity_scores)
    scores_df = pd.DataFrame(list(scores), index=df.index)
    scores_df["sentiment"] = scores_df["compound"].apply(_label)
    return pd.concat([df.reset_index(drop=True), scores_df.reset_index(drop=True)], axis=1)


if __name__ == "__main__":
    demo = pd.DataFrame({"text": [
        "I absolutely love this product, it's fantastic!",
        "Worst experience ever. Totally disappointing and awful.",
        "It arrived on Tuesday.",
        "Pretty good, but a bit boring in places.",
        "",
    ]})
    print(analyze_sentiment(demo, "text").to_string(index=False))


                                                   text      neg      neu      pos  compound sentiment
        I absolutely love this product, it's fantastic! 0.000000 0.714286 0.285714  0.666667  positive
Worst experience ever. Totally disappointing and awful. 0.428571 0.571429 0.000000 -0.750000  negative
                                 It arrived on Tuesday. 0.000000 1.000000 0.000000  0.000000   neutral
               Pretty good, but a bit boring in places. 0.125000 0.750000 0.125000  0.000000   neutral
                                                        0.000000 1.000000 0.000000  0.000000   neutral


In [3]:
"""
Simple sentiment analysis model.

Usage:
    from sentiment import analyze_sentiment
    import pandas as pd

    df = pd.DataFrame({"text": ["I love this!", "This is terrible.", "It's okay."]})
    result = analyze_sentiment(df, text_col="text")
    print(result)

Approach:
  - Lexicon-based scoring via VADER (works well on short/free text, no training needed).
  - Falls back to a tiny built-in lexicon if `vaderSentiment` isn't installed.
  - Returns the original df with added columns: neg, neu, pos, compound, sentiment.
"""

from __future__ import annotations
import re
from typing import Optional, Dict, Any, List, Union
import pandas as pd

__all__ = ["analyze_sentiment", "SentimentAnalyzer"]


class _FallbackAnalyzer:
    """Tiny lexicon fallback if vaderSentiment isn't installed."""

    # Core sentiment lexicons
    POS = {"good", "great", "love", "excellent", "amazing", "awesome", "happy", "best",
           "fantastic", "wonderful", "nice", "like", "enjoy", "perfect", "superb",
           "brilliant", "outstanding", "terrific", "magnificent", "exceptional",
           "delightful", "pleased", "glad", "favorable", "positive"}

    NEG = {"bad", "terrible", "hate", "awful", "worst", "horrible", "sad", "poor",
           "disappointing", "boring", "ugly", "dislike", "broken", "angry", "annoying",
           "mediocre", "lousy", "dreadful", "unpleasant", "horrendous", "abysmal",
           "frustrating", "irritating", "disgusting", "dismal", "appalling"}

    # Intensifiers (boost sentiment score)
    INTENSIFIERS = {
        "very": 1.5, "extremely": 2.0, "really": 1.3, "absolutely": 2.0,
        "totally": 1.5, "completely": 1.5, "incredibly": 1.8, "exceptionally": 1.7,
        "particularly": 1.4, "highly": 1.6, "so": 1.3, "quite": 1.2,
        "rather": 1.2, "pretty": 1.1, "somewhat": 0.8, "slightly": 0.7
    }

    # Negation words (flip sentiment)
    NEGATIONS = {"not", "never", "no", "n't", "neither", "nor", "hardly", "scarcely",
                 "barely", "without", "isn't", "aren't", "wasn't", "weren't",
                 "don't", "doesn't", "didn't", "won't", "wouldn't", "shouldn't",
                 "couldn't", "cannot"}

    def polarity_scores(self, text: str) -> Dict[str, float]:
        """
        Analyze sentiment with basic negation and intensifier handling.

        Args:
            text: Input text string

        Returns:
            Dictionary with neg, neu, pos, compound scores
        """
        if not text or not isinstance(text, str):
            return {"neg": 0.0, "neu": 1.0, "pos": 0.0, "compound": 0.0}

        # Tokenize with word boundaries
        words = re.findall(r"[a-zA-Z']+", text.lower())
        if not words:
            return {"neg": 0.0, "neu": 1.0, "pos": 0.0, "compound": 0.0}

        pos_score = 0.0
        neg_score = 0.0
        total_weight = 0.0
        i = 0

        while i < len(words):
            word = words[i]

            # Check for negations (affects following word)
            negation = 1.0
            j = i
            while j < len(words) and j < i + 3:  # Look ahead up to 3 words
                if words[j] in self.NEGATIONS:
                    negation = -1.0
                    j += 1
                    continue
                break
            i = j

            # Check for intensifiers
            intensifier = 1.0
            if j > 0 and words[j-1] in self.INTENSIFIERS:
                intensifier = self.INTENSIFIERS[words[j-1]]

            # Score the word
            if i < len(words):
                word = words[i]
                if word in self.POS:
                    pos_score += 1.0 * negation * intensifier
                    total_weight += 1.0
                elif word in self.NEG:
                    neg_score += 1.0 * negation * intensifier
                    total_weight += 1.0
                i += 1

        # Normalize scores
        total = len(words)
        if total == 0:
            return {"neg": 0.0, "neu": 1.0, "pos": 0.0, "compound": 0.0}

        # Clamp scores to [0, 1]
        p = max(0.0, min(1.0, pos_score / total if total > 0 else 0))
        n = max(0.0, min(1.0, neg_score / total if total > 0 else 0))
        neu = max(0.0, min(1.0, 1.0 - p - n))

        # Compound score with smoothing
        compound = (pos_score - neg_score) / (abs(pos_score) + abs(neg_score) + 1.0)
        compound = max(-1.0, min(1.0, compound))

        return {"neg": n, "neu": neu, "pos": p, "compound": compound}


def _get_analyzer():
    """Get VADER analyzer if available, otherwise fallback."""
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        return SentimentIntensityAnalyzer()
    except ImportError:
        return _FallbackAnalyzer()


def _label(compound: float, pos_threshold: float = 0.05, neg_threshold: float = -0.05) -> str:
    """
    Convert compound score to sentiment label.

    Args:
        compound: Compound sentiment score (-1 to 1)
        pos_threshold: Minimum compound score for positive classification
        neg_threshold: Maximum compound score for negative classification

    Returns:
        'positive', 'negative', or 'neutral'
    """
    if compound >= pos_threshold:
        return "positive"
    if compound <= neg_threshold:
        return "negative"
    return "neutral"


class SentimentAnalyzer:
    """
    Sentiment analyzer with caching and batch processing support.

    Usage:
        analyzer = SentimentAnalyzer()
        scores = analyzer.analyze_texts(["I love this!", "This is bad."])
        # Or with DataFrame:
        df = analyzer.analyze_dataframe(df, text_col="text")
    """

    def __init__(self, pos_threshold: float = 0.05, neg_threshold: float = -0.05):
        """
        Initialize the analyzer.

        Args:
            pos_threshold: Threshold for positive classification
            neg_threshold: Threshold for negative classification
        """
        self.analyzer = _get_analyzer()
        self.pos_threshold = pos_threshold
        self.neg_threshold = neg_threshold
        self._cache = {}  # Simple cache for repeated texts

    def analyze_text(self, text: Union[str, None]) -> Dict[str, Any]:
        """
        Analyze a single text string.

        Args:
            text: Input text

        Returns:
            Dictionary with sentiment scores and label
        """
        text_str = str(text) if text is not None else ""

        # Check cache
        if text_str in self._cache:
            return self._cache[text_str].copy()

        # Analyze
        scores = self.analyzer.polarity_scores(text_str)
        scores["sentiment"] = _label(
            scores["compound"],
            self.pos_threshold,
            self.neg_threshold
        )

        # Cache (limit size to prevent memory issues)
        if len(self._cache) < 10000:
            self._cache[text_str] = scores.copy()

        return scores

    def analyze_texts(self, texts: List[Union[str, None]],
                      show_progress: bool = False) -> pd.DataFrame:
        """
        Analyze multiple texts.

        Args:
            texts: List of text strings
            show_progress: Show progress bar (requires tqdm)

        Returns:
            DataFrame with sentiment scores for each text
        """
        if show_progress:
            try:
                from tqdm import tqdm
                iterator = tqdm(texts, desc="Analyzing sentiments")
            except ImportError:
                print("tqdm not installed. Install with: pip install tqdm")
                iterator = texts
        else:
            iterator = texts

        results = []
        for text in iterator:
            results.append(self.analyze_text(text))

        return pd.DataFrame(results)

    def analyze_dataframe(self, df: pd.DataFrame, text_col: str = "text",
                          show_progress: bool = False) -> pd.DataFrame:
        """
        Analyze sentiment for all rows in a DataFrame.

        Args:
            df: Input DataFrame
            text_col: Column containing text to analyze
            show_progress: Show progress bar

        Returns:
            Original DataFrame with sentiment columns added
        """
        if text_col not in df.columns:
            raise KeyError(f"Column '{text_col}' not found. Available: {list(df.columns)}")

        # Extract texts
        texts = df[text_col].fillna("").astype(str).tolist()

        # Analyze
        scores_df = self.analyze_texts(texts, show_progress=show_progress)
        scores_df.index = df.index

        # Add to original DataFrame
        result = df.reset_index(drop=True)
        scores_df = scores_df.reset_index(drop=True)

        return pd.concat([result, scores_df], axis=1)

    def clear_cache(self) -> None:
        """Clear the analysis cache."""
        self._cache.clear()


def analyze_sentiment(df: pd.DataFrame, text_col: str = "text",
                      pos_threshold: float = 0.05,
                      neg_threshold: float = -0.05,
                      show_progress: bool = False,
                      return_analyzer: bool = False) -> Union[pd.DataFrame, tuple[pd.DataFrame, SentimentAnalyzer]]:
    """
    Score each row's free text; returns df + neg/neu/pos/compound/sentiment.

    Args:
        df: Input DataFrame containing text to analyze
        text_col: Column name with text strings (default: 'text')
        pos_threshold: Minimum compound score for positive (default: 0.05)
        neg_threshold: Maximum compound score for negative (default: -0.05)
        show_progress: Show progress bar (requires tqdm)
        return_analyzer: If True, return (DataFrame, analyzer) for reuse

    Returns:
        Original DataFrame with 5 new columns:
        - neg: Negative sentiment score (0-1)
        - neu: Neutral sentiment score (0-1)
        - pos: Positive sentiment score (0-1)
        - compound: Overall normalized score (-1 to 1)
        - sentiment: Label ('positive', 'negative', 'neutral')

        If return_analyzer=True, returns (DataFrame, SentimentAnalyzer)

    Raises:
        KeyError: If text_col not found in DataFrame

    Examples:
        >>> df = pd.DataFrame({"text": ["I love this!", "Terrible!"]})
        >>> result = analyze_sentiment(df)
        >>> print(result[["text", "sentiment"]])

        >>> # With progress bar
        >>> result = analyze_sentiment(df, show_progress=True)

        >>> # Reuse analyzer for multiple DataFrames
        >>> result1, analyzer = analyze_sentiment(df1, return_analyzer=True)
        >>> result2 = analyzer.analyze_dataframe(df2)
    """
    analyzer = SentimentAnalyzer(pos_threshold=pos_threshold, neg_threshold=neg_threshold)
    result = analyzer.analyze_dataframe(df, text_col=text_col, show_progress=show_progress)

    if return_analyzer:
        return result, analyzer
    return result


def analyze_single_text(text: Union[str, None],
                        pos_threshold: float = 0.05,
                        neg_threshold: float = -0.05) -> Dict[str, Any]:
    """
    Analyze a single text string quickly.

    Args:
        text: Input text
        pos_threshold: Threshold for positive classification
        neg_threshold: Threshold for negative classification

    Returns:
        Dictionary with sentiment scores and label

    Example:
        >>> analyze_single_text("This is amazing!")
        {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 1.0, 'sentiment': 'positive'}
    """
    analyzer = SentimentAnalyzer(pos_threshold=pos_threshold, neg_threshold=neg_threshold)
    return analyzer.analyze_text(text)


def batch_analyze(texts: List[Union[str, None]],
                  pos_threshold: float = 0.05,
                  neg_threshold: float = -0.05,
                  show_progress: bool = False) -> pd.DataFrame:
    """
    Analyze a list of texts in batch.

    Args:
        texts: List of text strings to analyze
        pos_threshold: Threshold for positive classification
        neg_threshold: Threshold for negative classification
        show_progress: Show progress bar

    Returns:
        DataFrame with sentiment scores for each text

    Example:
        >>> texts = ["I love it!", "Terrible!", "Okay"]
        >>> batch_analyze(texts)
    """
    analyzer = SentimentAnalyzer(pos_threshold=pos_threshold, neg_threshold=neg_threshold)
    return analyzer.analyze_texts(texts, show_progress=show_progress)


if __name__ == "__main__":
    # Demo with various text types
    demo_data = {
        "text": [
            "I absolutely love this product, it's fantastic!",
            "Worst experience ever. Totally disappointing and awful.",
            "It arrived on Tuesday. The packaging was nice.",
            "Pretty good, but a bit boring in places.",
            "Not great, not terrible. Just average.",
            "I can't believe how amazing this is!",
            "",
            None,  # Test null handling
            "The service was not bad at all, actually quite good.",
        ]
    }

    demo_df = pd.DataFrame(demo_data)

    print("=" * 80)
    print("Sentiment Analysis Demo")
    print("=" * 80)

    # Using the main function
    result = analyze_sentiment(demo_df, "text", show_progress=True)
    print("\nResults:")
    print(result.to_string(index=False))

    print("\n" + "=" * 80)
    print("Summary Statistics:")
    print("=" * 80)
    print(result["sentiment"].value_counts().to_string())

    # Test single text analysis
    print("\n" + "=" * 80)
    print("Single Text Analysis Example:")
    print("=" * 80)
    single_result = analyze_single_text("This product is absolutely wonderful!")
    for key, value in single_result.items():
        print(f"  {key}: {value}")

    # Test batch analysis
    print("\n" + "=" * 80)
    print("Batch Analysis Example:")
    print("=" * 80)
    texts = ["Great!", "Not bad", "Terrible", "Average"]
    batch_result = batch_analyze(texts)
    for i, (text, scores) in enumerate(zip(texts, batch_result.to_dict('records'))):
        print(f"  '{text}' -> {scores['sentiment']} (compound: {scores['compound']:.2f})")

    # Test analyzer reuse
    print("\n" + "=" * 80)
    print("Analyzer Reuse Example:")
    print("=" * 80)
    result1, analyzer = analyze_sentiment(demo_df.iloc[:3], "text", return_analyzer=True)
    print("First batch:")
    print(result1[["text", "sentiment"]].to_string(index=False))

    # Reuse the same analyzer
    result2 = analyzer.analyze_dataframe(demo_df.iloc[3:6], "text")
    print("\nSecond batch (reusing analyzer):")
    print(result2[["text", "sentiment"]].to_string(index=False))

    # Show cache stats
    print(f"\nCache size: {analyzer._cache.__len__()} entries")

Sentiment Analysis Demo


Analyzing sentiments: 100%|██████████| 9/9 [00:00<00:00, 19630.13it/s]


Results:
                                                   text   neg      neu      pos  compound sentiment
        I absolutely love this product, it's fantastic! 0.000 0.571429 0.428571  0.750000  positive
Worst experience ever. Totally disappointing and awful. 0.500 0.500000 0.000000 -0.777778  negative
         It arrived on Tuesday. The packaging was nice. 0.000 0.875000 0.125000  0.500000  positive
               Pretty good, but a bit boring in places. 0.125 0.737500 0.137500  0.032258   neutral
                 Not great, not terrible. Just average. 0.000 1.000000 0.000000  0.000000   neutral
                   I can't believe how amazing this is! 0.000 0.857143 0.142857  0.500000  positive
                                                        0.000 1.000000 0.000000  0.000000   neutral
                                                   None 0.000 1.000000 0.000000  0.000000   neutral
   The service was not bad at all, actually quite good. 0.000 0.880000 0.120000  0.687500 